In [1]:
import os

import torch
import sqlite3
import pandas as pd

import json

from datetime import datetime

from pathlib import Path
import sys 

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

from src.config.paths import EMBEDS_DIR, EXPERIMENTS, RESULTS, DB_PATH, MODELS

time = datetime.now().strftime("%Y%m%d_%H%M%S")
MODEL_PATH = MODELS / "dino_adapter_block/20260801_221150"

EMBED_PATH = EMBEDS_DIR / "dino/20260801_221150"
EMBED_NAME = EMBED_PATH.stem

EXPERIMENTS_DIR = EXPERIMENTS / EMBED_NAME / f"ellipsoid_bootstrap/{time}"
RESULTS_DIR = RESULTS / EMBED_NAME / f"ellipsoid_bootstrap/{time}"

In [2]:
os.makedirs(EXPERIMENTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

In [3]:
cls_tokens = torch.load(EMBED_PATH/"cls.pt", weights_only=False)

In [4]:
conn = sqlite3.connect(DB_PATH)

meta = pd.read_sql_query("SELECT * FROM meta", conn)
categories = pd.read_sql_query("SELECT DISTINCT category FROM meta", conn)["category"].to_list()

conn.close()

In [ ]:
%load_ext autoreload
%autoreload 2

from src.algorithims.ellipsoid import EllipsoidCover, EllipsoidEvaluator, EllipsoidFitter, CandidateCleaner
from src.algorithims.ellipsoid.bootstrap import BootstrapRunner
from src.types import ExperimentConfig, AlgorithmResults

from src.stats.mahalanobis_detector import MahalanobisDetector

metadata = ExperimentConfig(
    K_frac=0.05,
    start_growth=1.2,
    min_growth=1,
    reg=1e-4,
    growth_type="variance_scaled",
    cleaner="shared_axis"
)

REG = 1e-4
fitter = EllipsoidFitter(support_points=5, reg=REG)
cleaner = CandidateCleaner(fitter=fitter, min_points=1)

cover = EllipsoidCover(fitter=fitter, cleaner=cleaner)
evaluator = EllipsoidEvaluator(reg=REG)

mal_detector = MahalanobisDetector(reg=1e-6) # smaller reg as its a larger matrix

In [6]:
with open(MODEL_PATH / "metadata.json", "r") as f:
    model_metadata = json.load(f)
    
neg_indices = model_metadata.get("negative_indices", [])

In [ ]:
test_df : list[pd.DataFrame] = []
train_df: list[pd.DataFrame] = []

for category in categories:
    print("Running", category)
    outputs_dir = EXPERIMENTS_DIR / category 
    os.makedirs(outputs_dir, exist_ok=True)

    train_mask = meta["split"] == "train"
    train_meta = meta[train_mask]
    test_meta = meta[
        (meta["split"] != "train")
        & (~meta.index.isin(neg_indices))
    ]

    train_cat_mask = train_meta["category"] == category

    train_emb = cls_tokens[train_mask]
    cat_emb = train_emb[train_cat_mask]

    good_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] == "good")
    defect_test_cat_mask = (test_meta["category"] == category) & (test_meta["type"] != "good")

    test_emb = cls_tokens[
        (meta["split"] != "train")
        & (~meta.index.isin(neg_indices))
    ]
    defect_test_emb = test_emb[defect_test_cat_mask]
    good_test_emb = test_emb[good_test_cat_mask]

    runner = BootstrapRunner(
        cover=cover,
        evaluator=evaluator,
        mal_detector=mal_detector,
        n_test_bootstraps=1000,
        n_train_bootstraps=100,
        seed=42,
    )

    test_bootstraps, train_bootstraps = runner.run(
        train_emb=cat_emb, 
        good_test_emb=good_test_emb, 
        defect_test_emb=defect_test_emb
        )

    train_bootstraps.insert(0, "category", category)
    test_bootstraps.insert(0, "category", category)

    train_bootstraps.to_csv(RESULTS_DIR / f"{category}_train_bootstraps.csv", index=False)
    test_bootstraps.to_csv(RESULTS_DIR / f"{category}_test_bootstraps.csv", index=False)

    test_df.append(test_bootstraps)
    train_df.append(train_bootstraps)

Running bottle


IndexError: boolean index did not match indexed array along axis 0; size of axis is 1725 but size of corresponding boolean axis is 1704

In [ ]:
train_summaries: list[pd.DataFrame] = []
test_summaries: list[pd.DataFrame] = []

for category, train_bootstraps, test_bootstraps in zip(
    categories,
    train_df,
    test_df,
):
    train_summary = runner.summarise_bootstrap(train_bootstraps)
    test_summary = runner.summarise_bootstrap(test_bootstraps)

    # In case summarise_bootstrap returns a Series
    if isinstance(train_summary, pd.Series):
        train_summary = train_summary.to_frame().T

    if isinstance(test_summary, pd.Series):
        test_summary = test_summary.to_frame().T

    train_summary.insert(0, "category", category)
    test_summary.insert(0, "category", category)

    train_summaries.append(train_summary)
    test_summaries.append(test_summary)

train_summary_df = pd.concat(
    train_summaries,
    ignore_index=True,
)

test_summary_df = pd.concat(
    test_summaries,
    ignore_index=True,
)

In [ ]:
train_summary_df.to_csv(
    RESULTS_DIR / "train_bootstrap_summary.csv",
    index=False,
)

train_summary_df

,category,alg_score_mean,alg_score_std,alg_score_ci_lower,alg_score_ci_upper,mal_score_mean,mal_score_std,mal_score_ci_lower,mal_score_ci_upper,delta_mean,...,pc1_ratio_mean_mean,pc1_ratio_mean_std,pc1_ratio_mean_ci_lower,pc1_ratio_mean_ci_upper,rank_mean_mean,rank_mean_std,rank_mean_ci_lower,rank_mean_ci_upper,mal_score,median_n_points
0,bottle,0.957548,0.029754,0.889643,0.988512,0.999944,0.000204,0.999206,1.000000,-0.042397,...,0.591181,0.019591,0.554319,0.627165,4.703770,0.357723,4.082317,5.434451,NaN,NaN
1,cable,0.765171,0.031916,0.675394,0.817012,0.918593,0.005696,0.908222,0.929367,-0.153422,...,0.557600,0.021690,0.519128,0.602507,5.151831,0.365189,4.450000,5.929762,NaN,NaN
2,capsule,0.794775,0.046021,0.674392,0.856402,0.950758,0.010073,0.931183,0.966095,-0.155983,...,0.593480,0.019786,0.551495,0.631926,5.008394,0.392668,4.343452,5.750595,NaN,NaN
3,carpet,0.993258,0.007579,0.971087,1.000000,0.993371,0.001185,0.990961,0.995395,-0.000112,...,0.549378,0.019129,0.518506,0.587254,6.276982,0.357972,5.635374,6.937199,NaN,NaN
4,grid,0.969616,0.028730,0.906266,0.997139,0.984348,0.001607,0.982456,0.988304,-0.014733,...,0.584932,0.017633,0.554864,0.613725,5.854952,0.430714,5.120290,6.659783,NaN,NaN
5,hazelnut,0.968750,0.011178,0.937804,0.982330,0.990229,0.002263,0.985714,0.993929,-0.021479,...,0.506296,0.016842,0.478773,0.546190,8.069760,0.438487,7.323685,8.830288,NaN,NaN
6,leather,0.952782,0.040106,0.835946,0.994599,NaN,NaN,NaN,NaN,-0.047218,...,0.582535,0.021104,0.542100,0.623977,5.518285,0.384301,4.769952,6.286932,1.0,NaN
7,metal_nut,0.849565,0.043463,0.740787,0.902761,0.992566,0.004222,0.983871,0.998534,-0.143001,...,0.596073,0.020920,0.553936,0.633313,5.068126,0.389910,4.384335,5.860061,NaN,NaN
8,pill,0.801746,0.045986,0.689546,0.865992,0.961776,0.005706,0.950675,0.972736,-0.160030,...,0.539851,0.017370,0.509698,0.575809,5.862557,0.412116,5.263623,6.791304,NaN,NaN
9,screw,0.847161,0.032834,0.783019,0.894866,0.921033,0.010189,0.900477,0.936063,-0.073872,...,0.558070,0.018132,0.522340,0.597766,6.985858,0.413789,6.180612,7.814509,NaN,NaN


In [ ]:
test_summary_df.to_csv(
    RESULTS_DIR / "test_bootstrap_summary.csv",
    index=False,
)

test_summary_df

,category,alg_score_mean,alg_score_std,alg_score_ci_lower,alg_score_ci_upper,mal_score_mean,mal_score_std,mal_score_ci_lower,mal_score_ci_upper,delta_mean,delta_std,delta_ci_lower,delta_ci_upper,mal_score
0,bottle,0.980972,0.012400,0.950794,0.998413,1.000000,1.405036e-17,1.000000,1.000000,-0.019028,0.012400,-4.920635e-02,-0.001587,NaN
1,cable,0.805113,0.033952,0.738943,0.869776,0.929725,1.855959e-02,0.888676,0.962153,-0.124612,0.024990,-1.778626e-01,-0.077956,NaN
2,capsule,0.817509,0.040418,0.729956,0.889120,0.962300,1.539073e-02,0.929388,0.987236,-0.144790,0.040046,-2.289589e-01,-0.072188,NaN
3,carpet,0.999616,0.000658,0.997592,1.000000,0.993982,6.245846e-03,0.978311,1.000000,0.005634,0.005882,-1.110223e-16,0.020465,NaN
4,grid,0.981716,0.017558,0.936508,1.000000,0.984314,1.582808e-02,0.946533,1.000000,-0.002598,0.003129,-1.004595e-02,0.001671,NaN
5,hazelnut,0.972795,0.012366,0.944643,0.992161,0.986958,7.882660e-03,0.967857,0.998929,-0.014164,0.009766,-3.500000e-02,0.001804,NaN
6,leather,0.995621,0.003657,0.986413,1.000000,NaN,NaN,NaN,NaN,-0.004379,0.003657,-1.358696e-02,0.000000,1.0
7,metal_nut,0.890351,0.031332,0.823045,0.944282,0.999043,1.369658e-03,0.995112,1.000000,-0.108692,0.031399,-1.754888e-01,-0.053751,NaN
8,pill,0.829934,0.039869,0.744108,0.902912,0.959211,1.422722e-02,0.929330,0.983640,-0.129277,0.033986,-1.942239e-01,-0.069013,NaN
9,screw,0.860512,0.030682,0.795245,0.912487,0.925713,2.570794e-02,0.871480,0.971521,-0.065201,0.033483,-1.363343e-01,-0.000179,NaN


: 